# EfficientNet-B0 Multilabel Shrimp Disease Pipeline

This notebook trains EfficientNet-B0 as a two-output multilabel classifier:

- output 0: BG probability
- output 1: WSSV probability

The original folders are mapped as:

- `1. Healthy` -> `[0, 0]` -> `Healthy`
- `2. BG` -> `[1, 0]` -> `BG`
- `3. WSSV` -> `[0, 1]` -> `WSSV`
- `4. WSSV_BG` -> `[1, 1]` -> `WSSV_BG`

The notebook uses the same data source, class folders, seed, split, image size, and ImageNet normalization conventions as `baseline-training-code.ipynb`. It then converts the model through ONNX -> SavedModel -> TFLite and evaluates both the PyTorch checkpoint and exported TFLite model on the held-out test split.

In [ ]:
# Colab dependencies. Restart the runtime if TensorFlow/onnx2tf package versions change.
%pip -q install torchmetrics onnx onnxruntime onnxslim onnxscript tensorflow==2.19.1 tf-keras==2.19.0 ai-edge-litert onnx2tf==1.28.8 onnx-graphsurgeon sng4onnx pandas pillow matplotlib seaborn tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

All outputs are written to a new folder so the previous softmax baselines stay untouched.

In [ ]:
import gc
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import torchvision.models as tv_models
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

try:
    from ai_edge_litert.interpreter import Interpreter
    INTERPRETER_BACKEND = 'ai_edge_litert'
except Exception:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
    INTERPRETER_BACKEND = 'tensorflow.lite'

import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Using TFLite interpreter backend: {INTERPRETER_BACKEND}')

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5
LR = 1e-4
NUM_WORKERS = 2
THRESHOLD_GRID = np.round(np.arange(0.10, 0.91, 0.05), 2)

DATA_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images')
OUTPUT_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/multilabel_effb0_pipeline')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
EXPORT_DIR = OUTPUT_DIR / 'exports'
SAVED_MODEL_DIR = OUTPUT_DIR / 'saved_models'
TFLITE_DIR = OUTPUT_DIR / 'tflite'
REPORT_DIR = OUTPUT_DIR / 'reports'
for d in [CHECKPOINT_DIR, EXPORT_DIR, SAVED_MODEL_DIR, TFLITE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ['1. Healthy', '2. BG', '3. WSSV', '4. WSSV_BG']
CLASS_NAMES = ['Healthy', 'BG', 'WSSV', 'WSSV_BG']
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}
DISEASE_NAMES = ['BG', 'WSSV']
MULTILABEL_MAP = {
    '1. Healthy': [0.0, 0.0],
    '2. BG': [1.0, 0.0],
    '3. WSSV': [0.0, 1.0],
    '4. WSSV_BG': [1.0, 1.0],
}
MODEL_KEY = 'efficientnet_b0_multilabel'

print(f'Output root: {OUTPUT_DIR}')

## Data Loading and Multilabel Targets

The stratified split still uses the original 4-class folder label so each split preserves the original class distribution.

In [ ]:
def class_name_from_multilabel(binary_vector):
    bg, wssv = [int(x) for x in binary_vector]
    if bg == 0 and wssv == 0:
        return 'Healthy'
    if bg == 1 and wssv == 0:
        return 'BG'
    if bg == 0 and wssv == 1:
        return 'WSSV'
    return 'WSSV_BG'

def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f'Warning: missing class folder: {folder}')
            continue
        for p in sorted(folder.iterdir()):
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                rows.append({
                    'path': str(p),
                    'class_dir': class_dir,
                    'class_label': CLASS_TO_IDX[class_dir],
                    'class_name': class_name_from_multilabel(MULTILABEL_MAP[class_dir]),
                    'bg_label': MULTILABEL_MAP[class_dir][0],
                    'wssv_label': MULTILABEL_MAP[class_dir][1],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No images found under {data_dir}. Run preprocessing first.')
    return df

df = discover_processed_images(DATA_DIR)
print(f'Loaded {len(df)} processed images from {DATA_DIR}')
print(df['class_dir'].value_counts().reindex(CLASS_DIRS))
print('Disease-positive counts:')
print(df[['bg_label', 'wssv_label']].sum())

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['class_label'],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df['class_label'],
    random_state=SEED,
    shuffle=True,
)

for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: {len(split_df)} images')
    print(split_df['class_dir'].value_counts().reindex(CLASS_DIRS).to_dict())
    print(split_df[['bg_label', 'wssv_label']].sum().to_dict())

train_paths = set(train_df['path'])
val_paths = set(val_df['path'])
test_paths = set(test_df['path'])
assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)
print('Image-level split overlap check passed.')

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

torch_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])

class ShrimpMultilabelDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        target = torch.tensor([row['bg_label'], row['wssv_label']], dtype=torch.float32)
        class_label = int(row['class_label'])
        return image, target, class_label

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    ShrimpMultilabelDataset(train_df, torch_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
val_loader = DataLoader(ShrimpMultilabelDataset(val_df, torch_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(ShrimpMultilabelDataset(test_df, torch_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_targets = train_df[['bg_label', 'wssv_label']].to_numpy(dtype=np.float32)
pos_counts = train_targets.sum(axis=0)
neg_counts = len(train_targets) - pos_counts
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32, device=device)
print('BCE pos_weight [BG, WSSV]:', pos_weight.detach().cpu().numpy())

## Train EfficientNet-B0 with BCEWithLogitsLoss

Checkpoint selection is based on validation decoded 4-class macro F1 after threshold tuning. The notebook searches `BG` and `WSSV` thresholds on the validation set, freezes the best pair, then evaluates the test split.

In [ ]:
def build_efficientnet_b0_multilabel():
    weights = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = tv_models.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(DISEASE_NAMES))
    return model.to(device)

def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

def decode_multilabel_probs(probabilities, thresholds):
    thresholds = np.asarray(thresholds, dtype=np.float32)
    binary = (probabilities >= thresholds).astype(int)
    return np.array([class_name_from_multilabel(row) for row in binary])

def collect_outputs(model, loader, criterion=None):
    model.eval()
    losses = []
    logits_list = []
    multilabel_targets = []
    class_targets = []
    with torch.no_grad():
        for ims, targets, class_labels in loader:
            ims = ims.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            logits = model(ims)
            if criterion is not None:
                losses.append(criterion(logits, targets).item() * ims.size(0))
            logits_list.append(logits.detach().cpu().numpy())
            multilabel_targets.append(targets.detach().cpu().numpy())
            class_targets.extend(class_labels.numpy().tolist())
    logits_np = np.concatenate(logits_list, axis=0)
    targets_np = np.concatenate(multilabel_targets, axis=0)
    class_true = np.array([CLASS_NAMES[i] for i in class_targets])
    return {
        'loss': sum(losses) / max(1, len(class_true)) if criterion is not None else None,
        'logits': logits_np,
        'probabilities': sigmoid_np(logits_np),
        'multilabel_targets': targets_np,
        'class_true': class_true,
    }

def disease_level_metrics(y_true_binary, y_prob, thresholds):
    y_pred_binary = (y_prob >= np.asarray(thresholds)).astype(int)
    rows = []
    for idx, disease in enumerate(DISEASE_NAMES):
        p, r, f, support = precision_recall_fscore_support(
            y_true_binary[:, idx],
            y_pred_binary[:, idx],
            average='binary',
            zero_division=0,
        )
        try:
            auc = roc_auc_score(y_true_binary[:, idx], y_prob[:, idx])
        except ValueError:
            auc = np.nan
        rows.append({
            'disease': disease,
            'precision': p,
            'recall': r,
            'f1_score': f,
            'support': support,
            'threshold': thresholds[idx],
            'roc_auc': auc,
        })
    return pd.DataFrame(rows)

def decoded_class_summary(class_true, class_pred):
    return {
        'accuracy': accuracy_score(class_true, class_pred),
        'macro_f1': f1_score(class_true, class_pred, labels=CLASS_NAMES, average='macro', zero_division=0),
        'cohen_kappa': cohen_kappa_score(class_true, class_pred, labels=CLASS_NAMES),
    }

def per_class_report_df(model_key, class_true, class_pred):
    report = classification_report(class_true, class_pred, labels=CLASS_NAMES, output_dict=True, zero_division=0)
    rows = []
    for class_name in CLASS_NAMES:
        rows.append({
            'model_key': model_key,
            'class': class_name,
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1_score': report[class_name]['f1-score'],
            'support': report[class_name]['support'],
        })
    return pd.DataFrame(rows)

def tune_thresholds(class_true, y_prob):
    best = {'macro_f1': -1.0, 'thresholds': (0.5, 0.5), 'accuracy': np.nan, 'cohen_kappa': np.nan}
    records = []
    for t_bg in THRESHOLD_GRID:
        for t_wssv in THRESHOLD_GRID:
            thresholds = (float(t_bg), float(t_wssv))
            class_pred = decode_multilabel_probs(y_prob, thresholds)
            summary = decoded_class_summary(class_true, class_pred)
            record = {'threshold_bg': thresholds[0], 'threshold_wssv': thresholds[1], **summary}
            records.append(record)
            if summary['macro_f1'] > best['macro_f1']:
                best = {'thresholds': thresholds, **summary}
    return best, pd.DataFrame(records).sort_values('macro_f1', ascending=False).reset_index(drop=True)

def evaluate_with_thresholds(model_key, outputs, thresholds, elapsed=None):
    class_pred = decode_multilabel_probs(outputs['probabilities'], thresholds)
    summary = decoded_class_summary(outputs['class_true'], class_pred)
    summary.update({
        'model_key': model_key,
        'n_images': len(class_pred),
        'threshold_bg': thresholds[0],
        'threshold_wssv': thresholds[1],
    })
    if elapsed is not None:
        summary['total_inference_time_s'] = elapsed
        summary['mean_latency_ms'] = (elapsed / len(class_pred)) * 1000
        summary['fps'] = len(class_pred) / elapsed if elapsed > 0 else np.nan
    predictions = pd.DataFrame({
        'model_key': model_key,
        'true_label': outputs['class_true'],
        'predicted_label': class_pred,
        'prob_BG': outputs['probabilities'][:, 0],
        'prob_WSSV': outputs['probabilities'][:, 1],
    })
    per_class = per_class_report_df(model_key, outputs['class_true'], class_pred)
    disease = disease_level_metrics(outputs['multilabel_targets'], outputs['probabilities'], thresholds)
    disease.insert(0, 'model_key', model_key)
    return summary, predictions, per_class, disease

In [ ]:
def train_model():
    model = build_efficientnet_b0_multilabel()
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    best_f1 = -1.0
    best_thresholds = (0.5, 0.5)
    epochs_no_improve = 0
    best_path = CHECKPOINT_DIR / f'best_{MODEL_KEY}.pth'
    history = []
    train_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_total = 0
        for ims, targets, _ in tqdm(train_loader, desc=f'{MODEL_KEY} epoch {epoch + 1}/{EPOCHS}', leave=False):
            ims = ims.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            logits = model(ims)
            loss = criterion(logits, targets)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * ims.size(0)
            train_total += ims.size(0)

        val_outputs = collect_outputs(model, val_loader, criterion=criterion)
        best_val, threshold_table = tune_thresholds(val_outputs['class_true'], val_outputs['probabilities'])
        threshold_table.to_csv(REPORT_DIR / f'threshold_search_epoch_{epoch + 1:02d}.csv', index=False)
        epoch_record = {
            'epoch': epoch + 1,
            'train_loss': train_loss / max(1, train_total),
            'val_loss': val_outputs['loss'],
            'val_accuracy': best_val['accuracy'],
            'val_macro_f1': best_val['macro_f1'],
            'val_cohen_kappa': best_val['cohen_kappa'],
            'threshold_bg': best_val['thresholds'][0],
            'threshold_wssv': best_val['thresholds'][1],
        }
        history.append(epoch_record)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {epoch_record['train_loss']:.4f} | "
            f"Val Loss: {epoch_record['val_loss']:.4f} | "
            f"Val Macro F1: {epoch_record['val_macro_f1']:.4f} | "
            f"Thresholds BG/WSSV: {best_val['thresholds']}"
        )

        if best_val['macro_f1'] > best_f1:
            best_f1 = best_val['macro_f1']
            best_thresholds = best_val['thresholds']
            epochs_no_improve = 0
            torch.save({'model_state_dict': model.state_dict(), 'thresholds': best_thresholds, 'best_val': best_val}, best_path)
            print(f'  --> Saved best checkpoint: macro F1 {best_f1:.4f}')
        else:
            epochs_no_improve += 1
            print(f'  --> No improvement ({epochs_no_improve}/{PATIENCE})')

        if epochs_no_improve >= PATIENCE:
            print('  --> Early stopping triggered.')
            break

    train_time = time.time() - train_start
    checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_thresholds = tuple(checkpoint['thresholds'])
    pd.DataFrame(history).to_csv(REPORT_DIR / f'{MODEL_KEY}_training_history.csv', index=False)
    return model, best_path, best_thresholds, train_time

model, best_checkpoint_path, selected_thresholds, training_time_s = train_model()
print('Best checkpoint:', best_checkpoint_path)
print('Selected thresholds:', selected_thresholds)

## PyTorch Test Evaluation

This evaluates the best PyTorch checkpoint using the validation-selected thresholds. These numbers are useful for training diagnostics; the later TFLite evaluation is the deployment-facing result.

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Timed PyTorch test collection.
model.eval()
if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.perf_counter()
test_outputs = collect_outputs(model, test_loader, criterion=criterion)
if torch.cuda.is_available():
    torch.cuda.synchronize()
pytorch_elapsed = time.perf_counter() - start

pytorch_summary, pytorch_predictions, pytorch_per_class, pytorch_disease_metrics = evaluate_with_thresholds(
    'pytorch_' + MODEL_KEY,
    test_outputs,
    selected_thresholds,
    elapsed=pytorch_elapsed,
)
pytorch_summary.update({
    'training_time_s': training_time_s,
    'checkpoint_path': str(best_checkpoint_path),
    'test_loss': test_outputs['loss'],
})

pytorch_summary_df = pd.DataFrame([pytorch_summary])
pytorch_summary_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_pytorch_test_summary.csv', index=False)
pytorch_predictions.to_csv(REPORT_DIR / f'{MODEL_KEY}_pytorch_test_predictions.csv', index=False)
pytorch_per_class.to_csv(REPORT_DIR / f'{MODEL_KEY}_pytorch_per_class_metrics.csv', index=False)
pytorch_disease_metrics.to_csv(REPORT_DIR / f'{MODEL_KEY}_pytorch_disease_metrics.csv', index=False)

display(pytorch_summary_df)
display(pytorch_per_class)
display(pytorch_disease_metrics)

## Convert PyTorch Model to TFLite

The conversion path follows the existing conversion notebook: PyTorch -> ONNX -> TensorFlow SavedModel -> TFLite.

In [ ]:
def run_command(cmd, cwd=None):
    print('Running:', ' '.join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout[-4000:])
    if completed.stderr:
        print(completed.stderr[-4000:])
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {cmd}')
    return completed

def export_to_onnx(model):
    model_cpu = model.to('cpu').eval()
    sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    onnx_path = EXPORT_DIR / f'{MODEL_KEY}.onnx'
    torch.onnx.export(
        model_cpu,
        sample,
        str(onnx_path),
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17,
        dynamo=False,
    )
    model.to(device)
    print(f'Saved ONNX: {onnx_path} ({onnx_path.stat().st_size / (1024 * 1024):.2f} MB)')
    return onnx_path

def convert_onnx_to_saved_model(onnx_path):
    out_dir = SAVED_MODEL_DIR / onnx_path.stem
    if out_dir.exists():
        shutil.rmtree(out_dir)
    run_command([
        sys.executable, '-m', 'onnx2tf',
        '-i', str(onnx_path),
        '-o', str(out_dir),
        '-osd',
    ])
    candidates = [p for p in out_dir.rglob('saved_model.pb')]
    if (out_dir / 'saved_model.pb').exists():
        saved_model_dir = out_dir
    elif candidates:
        saved_model_dir = candidates[0].parent
    else:
        raise FileNotFoundError(f'No saved_model.pb found under {out_dir}')
    print(f'Saved TensorFlow SavedModel: {saved_model_dir}')
    return saved_model_dir

def convert_saved_model_to_tflite(saved_model_path, variant='float32'):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_path))
    if variant == 'float32':
        pass
    elif variant == 'dynamic_range':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variant == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    else:
        raise ValueError(f'Unsupported TFLite variant: {variant}')
    tflite_bytes = converter.convert()
    out_path = TFLITE_DIR / f'{MODEL_KEY}_{variant}.tflite'
    out_path.write_bytes(tflite_bytes)
    print(f'Saved {variant} TFLite: {out_path} ({out_path.stat().st_size / (1024 * 1024):.2f} MB)')
    return out_path

onnx_path = export_to_onnx(model)
saved_model_path = convert_onnx_to_saved_model(onnx_path)

conversion_records = []
for variant in ['float32', 'float16']:
    try:
        t0 = time.time()
        tflite_path = convert_saved_model_to_tflite(saved_model_path, variant=variant)
        conversion_records.append({
            'model_key': f'{MODEL_KEY}_{variant}',
            'variant': variant,
            'status': 'converted',
            'path': str(tflite_path),
            'size_mb': round(tflite_path.stat().st_size / (1024 * 1024), 2),
            'seconds': round(time.time() - t0, 1),
            'error': '',
        })
    except Exception as exc:
        conversion_records.append({
            'model_key': f'{MODEL_KEY}_{variant}',
            'variant': variant,
            'status': 'failed',
            'path': '',
            'size_mb': np.nan,
            'seconds': np.nan,
            'error': f'{type(exc).__name__}: {exc}',
        })

conversion_df = pd.DataFrame(conversion_records)
conversion_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_conversion_report.csv', index=False)
display(conversion_df)

## TFLite Test Evaluation

The TFLite model returns two logits. The notebook applies sigmoid, decodes with the frozen validation-selected thresholds, and reports both decoded 4-class metrics and disease-level multilabel metrics.

In [ ]:
def load_interpreter(model_path):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    return interpreter, interpreter.get_input_details()[0], interpreter.get_output_details()[0]

def infer_hw_from_input_shape(input_shape):
    shape = [int(x) for x in input_shape]
    if len(shape) != 4:
        return IMG_SIZE, IMG_SIZE, 'NHWC'
    if shape[1] == 3:
        return shape[2], shape[3], 'NCHW'
    return shape[1], shape[2], 'NHWC'

def preprocess_for_tflite(image_path, input_details):
    input_shape = input_details['shape']
    input_dtype = input_details['dtype']
    height, width, layout = infer_hw_from_input_shape(input_shape)
    image = Image.open(image_path).convert('RGB').resize((width, height))
    x = np.asarray(image, dtype=np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    if layout == 'NCHW':
        x = np.transpose(x, (2, 0, 1))
    x = np.expand_dims(x, axis=0)
    if np.issubdtype(input_dtype, np.integer):
        scale, zero_point = input_details.get('quantization', (0.0, 0))
        if scale and scale > 0:
            x = x / scale + zero_point
        info = np.iinfo(input_dtype)
        x = np.clip(np.rint(x), info.min, info.max).astype(input_dtype)
    else:
        x = x.astype(input_dtype)
    return x

def dequantize_output(y, output_details):
    y = np.asarray(y)
    if np.issubdtype(y.dtype, np.integer):
        scale, zero_point = output_details.get('quantization', (0.0, 0))
        if scale and scale > 0:
            y = (y.astype(np.float32) - zero_point) * scale
    return y

def evaluate_tflite_multilabel(model_key, model_path, eval_df, thresholds):
    interpreter, input_details, output_details = load_interpreter(model_path)
    warmup_x = preprocess_for_tflite(eval_df.iloc[0]['path'], input_details)
    interpreter.set_tensor(input_details['index'], warmup_x)
    interpreter.invoke()

    logits_rows = []
    prob_rows = []
    binary_targets = []
    true_labels = []
    image_paths = []
    latencies = []

    total_start = time.perf_counter()
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=model_key, leave=False):
        x = preprocess_for_tflite(row['path'], input_details)
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details['index'], x)
        interpreter.invoke()
        latencies.append((time.perf_counter() - t0) * 1000)
        logits = dequantize_output(interpreter.get_tensor(output_details['index']), output_details).reshape(-1).astype(np.float32)
        probs = sigmoid_np(logits)
        logits_rows.append(logits)
        prob_rows.append(probs)
        binary_targets.append([row['bg_label'], row['wssv_label']])
        true_labels.append(row['class_name'])
        image_paths.append(row['path'])
    elapsed = time.perf_counter() - total_start

    outputs = {
        'logits': np.vstack(logits_rows),
        'probabilities': np.vstack(prob_rows),
        'multilabel_targets': np.asarray(binary_targets, dtype=np.float32),
        'class_true': np.asarray(true_labels),
    }
    summary, predictions, per_class, disease = evaluate_with_thresholds(model_key, outputs, thresholds, elapsed=elapsed)
    predictions.insert(1, 'image_path', image_paths)
    predictions['latency_ms'] = latencies
    predictions['prob_BG'] = outputs['probabilities'][:, 0]
    predictions['prob_WSSV'] = outputs['probabilities'][:, 1]
    summary['model_size_mb'] = round(Path(model_path).stat().st_size / (1024 * 1024), 2)
    summary['path'] = str(model_path)
    return summary, predictions, per_class, disease

summaries = []
prediction_frames = []
per_class_frames = []
disease_frames = []
for _, row in conversion_df[conversion_df['status'] == 'converted'].iterrows():
    summary, predictions, per_class, disease = evaluate_tflite_multilabel(
        row['model_key'],
        Path(row['path']),
        test_df,
        selected_thresholds,
    )
    summaries.append(summary)
    prediction_frames.append(predictions)
    per_class_frames.append(per_class)
    disease_frames.append(disease)

tflite_summary_df = pd.DataFrame(summaries).sort_values('macro_f1', ascending=False).reset_index(drop=True)
tflite_predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
tflite_per_class_df = pd.concat(per_class_frames, ignore_index=True) if per_class_frames else pd.DataFrame()
tflite_disease_df = pd.concat(disease_frames, ignore_index=True) if disease_frames else pd.DataFrame()

tflite_summary_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_test_summary.csv', index=False)
tflite_predictions_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_test_predictions.csv', index=False)
tflite_per_class_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_per_class_metrics.csv', index=False)
tflite_disease_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_disease_metrics.csv', index=False)

display(tflite_summary_df)
display(tflite_per_class_df)
display(tflite_disease_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for model_key in ['pytorch_' + MODEL_KEY] + tflite_summary_df['model_key'].dropna().tolist():
    if model_key == 'pytorch_' + MODEL_KEY:
        preds = pytorch_predictions
    else:
        preds = tflite_predictions_df[tflite_predictions_df['model_key'] == model_key]
    if preds.empty:
        continue
    cm = confusion_matrix(preds['true_label'], preds['predicted_label'], labels=CLASS_NAMES)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
    plt.title(model_key)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    fig_path = REPORT_DIR / f'confusion_matrix_{model_key}.png'
    plt.savefig(fig_path, dpi=160)
    plt.show()
    print(f'Saved: {fig_path}')

## Result Artifacts

Outputs are saved under:

`/content/drive/MyDrive/shrimp_disease_images/multilabel_effb0_pipeline`

Key files:

- `reports/efficientnet_b0_multilabel_training_history.csv`
- `reports/efficientnet_b0_multilabel_pytorch_test_summary.csv`
- `reports/efficientnet_b0_multilabel_pytorch_per_class_metrics.csv`
- `reports/efficientnet_b0_multilabel_pytorch_disease_metrics.csv`
- `exports/efficientnet_b0_multilabel.onnx`
- `tflite/efficientnet_b0_multilabel_float32.tflite`
- `tflite/efficientnet_b0_multilabel_float16.tflite`
- `reports/efficientnet_b0_multilabel_tflite_test_summary.csv`
- `reports/efficientnet_b0_multilabel_tflite_per_class_metrics.csv`
- `reports/efficientnet_b0_multilabel_tflite_disease_metrics.csv`

Compare the decoded 4-class TFLite `macro_f1` and per-class recall/precision against the current YOLO softmax model. Also inspect disease-level `BG` and `WSSV` recall, because this is the main reason to try multilabel training.